# Policy search — convergence to formal concepts

Tests `streamlined/policy.py`: the push/replace/pop path search driven by
`FixpointIterator`.

The claim: if the energy function is right, `search()` should converge to a
path whose output is a **formal concept** of the relation — a fixed point where
energy = 0 and the state stops changing.

Ground truth: we precompute all formal concepts using `_concept_fixpoint` directly,
then check whether `search()` recovers one of them from different seeds.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import torch

from streamlined.path_engine import PathEngine
from streamlined.relational_einsum import join_einsum_forward, residuate_einsum_forward
from streamlined.policy import search, _step, _energy
from core.fixpoint import FixpointIterator

engine = PathEngine({
    "realize":   (join_einsum_forward,      "j,ji->i", lambda x, y: (x, y.T)),
    "propagate": (join_einsum_forward,      "i,ij->j", lambda x, y: (x, y)),
    "abstract":  (residuate_einsum_forward, "ij,i->j", lambda x, y: (y, x)),
    "support":   (residuate_einsum_forward, "ji,j->i", lambda x, y: (y.T, x)),
})

print('imports ok')
print('operator types:', {n: f'{i}->{o}' for n, (i, o) in engine._types.items()})

## 1. Build a non-degenerate relation

5×5 asymmetric R. No single row is a fixpoint of the single-leg ops,
so the policy actually has to search. The formal concepts are the fixed
points of `support propagate` (extent closure) — we verify the policy
finds that path from arbitrary seeds.

In [ ]:
R = torch.tensor([
    [1., 1., 0., 0., 1.],
    [1., 1., 1., 0., 0.],
    [0., 1., 1., 1., 0.],
    [0., 0., 1., 1., 0.],
    [1., 0., 0., 0., 1.],
], dtype=torch.float32)

y = R.clone()

print('R:')
print(R.numpy())

## 2. Ground truth: formal concepts via extent closure

The formal concepts are fixed points of `support propagate` at temp=0.
We seed from each row of R and collect the distinct stable extents.

In [ ]:
# Ground truth: run "support propagate" to fixpoint from each row seed
# This is the extent-closure operator — its fixed points ARE the formal concepts.

concepts = {}  # key: rounded tuple, value: tensor

for i in range(R.shape[0]):
    x = R[i].clone()
    # Iterate support∘propagate until stable (converges in ≤2 steps by Belohlávek)
    for _ in range(10):
        x_new = engine.run("support propagate", x, y, temp=0.0)
        if torch.allclose(x_new, x, atol=1e-4):
            break
        x = x_new
    key = tuple(x.round().int().tolist())
    if key not in concepts:
        concepts[key] = x.clone()
        print(f'concept from row {i}: {x.numpy().round(2)}')

print(f'\n{len(concepts)} distinct formal concepts')

## 3. Run policy search from each row seed

Seed the search with `"realize"` (non-zero energy for this R).
Check: does the policy find `"support propagate"` — the path whose
fixed points are the formal concepts?

In [ ]:
eps = 1e-2

for i in range(R.shape[0]):
    x_i = R[i].clone()

    best_spec, final_energy = search(
        engine, x_i, y,
        seed="realize",
        temp=1.0,
        eps=eps,
        max_iters=50,
        verbose=False,
    )

    z = engine.run(best_spec, x_i, y, temp=0.0)
    matched = any(torch.allclose(z, c, atol=0.05) for c in concepts.values())

    print(f'row {i}  spec={best_spec!r:28s}  energy={final_energy:.6f}  concept_match={matched}')
    print(f'       z={z.numpy().round(2)}')

## 5. Vary the seed path

Try different single-leg seeds — the algorithm should converge to the same
formal concepts regardless of which leg we start from.

In [ ]:
x_test = R[0].clone()

for seed in ["realize", "propagate", "abstract", "support"]:
    try:
        _ = engine.run(seed, x_test, y, temp=1.0)
    except Exception as e:
        print(f'seed {seed!r}: invalid — {e}')
        continue

    best_spec, final_energy = search(
        engine, x_test, y,
        seed=seed,
        temp=1.0,
        eps=eps,
        max_iters=50,
    )
    z = engine.run(best_spec, x_test, y, temp=0.0)
    matched = any(torch.allclose(z, c, atol=0.05) for c in concepts.values())
    print(f'seed={seed!r:12s}  converged={best_spec!r:28s}  energy={final_energy:.6f}  match={matched}')

## 6. Energy trace

Plot energy over iterations for one run to see the annealing dynamic.

In [ ]:
x_test = R[0].clone()
spec_cell = ["realize"]
state0 = np.array([_energy(engine, "realize", x_test, y, temp=1.0)])

energies = [state0[0]]
specs    = ["realize"]
temps    = [1.0]

fp = FixpointIterator(
    f=lambda state, t: _step(state, t, engine=engine, spec_cell=spec_cell, x=x_test, y=y),
    state0=state0,
    temp=1.0,
    eps=eps,
    max_iters=50,
)

for _ in range(50):
    converged = fp.step()
    energies.append(fp.energy)
    specs.append(spec_cell[0])
    temps.append(fp.temp)
    if converged:
        break

print(f'Converged at iter {len(energies)-1},  final spec: {spec_cell[0]!r}')
print()
print(f'{"iter":>4}  {"spec":28}  {"energy":>12}  {"temp":>10}')
for i, (s, e, t) in enumerate(zip(specs, energies, temps)):
    print(f'{i:4d}  {s:28}  {e:12.6f}  {t:10.6f}')